<a href="https://colab.research.google.com/github/EMuthomi-dev/Introduction-to-ML/blob/Assignments/Problem_framing_pothole_detection_(Week_3).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pothole Detection and Severity Classification

## Project Problem
Road agencies rely on manual inspections and slow public reports to find potholes, meaning the most dangerous ones are repaired late. This causes crashes, vehicle damage, and higher long-term repair costs.

## Intended AI/ML Outcome
A computer vision model that takes a road image, detects whether a pothole is present, and classifies its severity (Low / Medium / High), so road repairs can be prioritized automatically.

## Data Needed

| Data Type | Purpose |
| :--- | :--- |
| Road images (dashboard, phone, or drone) with and without potholes | Model input features |
| Label per image: pothole / no pothole | Detection target |
| Severity label per pothole (Low / Medium / High), ideally derived from measured depth (cm) and width (cm) | Severity target |
| Optional: Bounding boxes or segmentation masks | Localization |
| Variety in lighting, wet/dry roads, shadows, and asphalt surface types | Generalization and robustness |

**Dataset Choice:**
The Hugging Face dataset `taroii/pothole-detection` (public, image-based, fewer than 1K images) is used as a stand-in baseline since it loads instantly in Colab. *Note:* It may lack specific severity labels, which will be addressed in future project iterations.

In [ ]:
# Install datasets library if not already present
!pip -q install datasets

import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset

# 1. Load the dataset
ds = load_dataset("taroii/pothole-detection")
print(ds)

# 2. Inspect structure
split = "train" if "train" in ds else list(ds.keys())[0]
d = ds[split]
print("\nSplit used:", split)
print("Columns / features:", d.features)

# 3. Class names and counts per split
label_names = d.features["label"].names if "label" in d.features else None
print("Class names:", label_names)

rows = []
for s in ds.keys():
    counts = pd.Series(ds[s]["label"]).value_counts().sort_index()
    for idx, n in counts.items():
        rows.append({
            "split": s,
            "class": label_names[idx] if label_names else idx,
            "images": n
        })
summary = pd.DataFrame(rows)
display(summary)

# 4. Display a small random portion of the data
random.seed(42)
sample_ids = random.sample(range(len(d)), 8)
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, i in zip(axes.ravel(), sample_ids):
    ex = d[i]
    ax.imshow(ex["image"])
    ax.set_title(label_names[ex["label"]] if label_names else ex["label"])
    ax.axis("off")
plt.suptitle("Random sample from the dataset")
plt.tight_layout()
plt.show()

# 5. Image size check (inputs are not all the same size)
sizes = [d[i]["image"].size for i in sample_ids]
print("Sample image sizes (width, height) in px:", sizes)

## Inputs (Features) and Outputs (Target)

**Inputs (Features):**
* Raw RGB road images, resized to a standardized dimension (e.g., $224 \times 224$ pixels) and normalized. Individual pixel values serve as raw features, from which a CNN extracts higher-level representations (edges, textures, shapes).
* Potential environmental metadata if available: capture location coordinates, time of day, and weather conditions.

**Outputs (Target):**
* *Current dataset target:* Binary or categorical class label (pothole vs. no pothole).
* *Full project target:* Severity class (`Low`, `Medium`, or `High`), ideally mapped from actual physical depth and volume measurements.

**Task Type:** Supervised image classification (with bounding box localization as an advanced extension).  
**Objective Function:** Minimize cross-entropy loss.  
**Evaluation Metrics:** Accuracy, macro-F1 score, and recall specifically targeted at high-severity road hazards.

## Proposed ML Workflow

```text
[1. Collect road images + labels]
              │
              ▼
[2. Explore and clean data] ─ check class balance, remove corrupt files
              │
              ▼
[3. Split dataset] ───────── train (70%) / validation (15%) / test (15%)
              │
              ▼
[4. Preprocess + augment] ── resize, normalize, flip, adjust brightness
              │
              ▼
[5. Train model] ─────────── transfer learning using a pretrained CNN backbone
              │
              ▼
[6. Evaluate performance] ── accuracy, F1-score, confusion matrix analysis
              │
              ▼
[7. Iterate & improve] ───── hyperparameter tuning, data expansion ──┐
              │                                                    │
              ▼                                                    │
[8. Deploy & monitor] ────── dashboard for road crews & field tracking ┘
```

##### Workflow Breakdown

1. **Collect:** Gather images and corresponding hazard tags (public data currently, field captures later).
2. **Explore and Clean:** Inspect class distribution and filter out corrupted files.
3. **Split:** Carve out an untouched test set to ensure honest final evaluation.
4. **Preprocess & Augment:** Standardize dimensions and introduce lighting/weather variations.
5. **Train:** Leverage pre-trained convolutional neural networks to compensate for small initial dataset sizes.
6. **Evaluate:** Use confusion matrices to identify where the model confuses different severity levels.
7. **Improve:** Address weaknesses found during evaluation through targeted data collection.
8. **Deploy & Monitor:** Integrate into a mobile/web dashboard for municipal road teams and track performance drift over time.

## Limitations & Future Work

* **Dataset Scale & Label Constraints:** The public baseline dataset is relatively small and lacks precise physical depth or structured severity labels.
* **Environmental Variability:** Variations in shadow patterns, heavy rain, and night driving can obscure road damage features.
* **Path Forward:** The full production project will incorporate custom field-collected imagery featuring physically measured depths (in centimeters) and standardized multi-class severity thresholds.